# Стандартизация и нормализация

**Шаг 5 плана.** StandardScaler (и опция MinMaxScaler), fit на train, transform на train/val/test для финального target `tp_sl_1_05`. Сохранение scaler.

## 1. Импорты и загрузка

In [1]:
import sys
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) in ('01_data_prep','02_targets','03_features','04_models','05_experiments') else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)

TARGET_COL = 'target'
df = pd.read_parquet(os.path.join(_root, 'outputs', 'data_labeled_tp_sl_1_05.parquet'))

feat_path = os.path.join(_root, 'outputs', 'features_selected_tp_sl_1_05.txt')
if not os.path.exists(feat_path):
    raise FileNotFoundError(
        f'Запустите 06_Feature_Selection.ipynb. Файл не найден: {feat_path}'
    )

with open(feat_path, encoding='utf-8') as f:
    FEATURES_SELECTED = [l.strip() for l in f if l.strip()]

missing = [c for c in FEATURES_SELECTED if c not in df.columns]
if missing:
    raise ValueError(f'В датасете отсутствуют колонки из features_selected: {missing}')

print(f'Фичей: {len(FEATURES_SELECTED)}')

Фичей: 22


## 2. Train/Val/Test split

In [2]:
valid = df.dropna(subset=FEATURES_SELECTED + [TARGET_COL]).copy()
valid = valid[valid[TARGET_COL].isin([-1.0, 1.0])]

sessions = list(valid['session_key'].unique())
np.random.seed(42)
np.random.shuffle(sessions)
n = len(sessions)
n_train, n_val = int(0.7 * n), int(0.15 * n)
train_sessions = set(sessions[:n_train])
val_sessions = set(sessions[n_train:n_train + n_val])
test_sessions = set(sessions[n_train + n_val:])

train_df = valid[valid['session_key'].isin(train_sessions)]
val_df = valid[valid['session_key'].isin(val_sessions)]
test_df = valid[valid['session_key'].isin(test_sessions)]

print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

Train: 129,432 | Val: 29,020 | Test: 28,652


## 3. StandardScaler

In [3]:
scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[FEATURES_SELECTED].fillna(0))
X_val = scaler.transform(val_df[FEATURES_SELECTED].fillna(0))
X_test = scaler.transform(test_df[FEATURES_SELECTED].fillna(0))

def _stats(X, name):
    m, s = X.mean(axis=0), X.std(axis=0)
    return f'{name}: mean∈[{m.min():.3f}, {m.max():.3f}], std∈[{s.min():.3f}, {s.max():.3f}]'

print('Проверка StandardScaler (fit на train, transform на всех):')
print('  ' + _stats(X_train, 'Train'))
print('  ' + _stats(X_val, 'Val'))
print('  ' + _stats(X_test, 'Test'))
print('  Train: mean≈0, std≈1. Val/Test: иные значения — ожидаемо, утечки нет.')

Проверка StandardScaler (fit на train, transform на всех):
  Train: mean∈[-0.000, 0.000], std∈[1.000, 1.000]
  Val: mean∈[-0.118, 0.142], std∈[0.464, 1.047]
  Test: mean∈[-0.041, 0.022], std∈[0.781, 1.563]
  Train: mean≈0, std≈1. Val/Test: иные значения — ожидаемо, утечки нет.


## 4. Сравнение StandardScaler vs MinMaxScaler (LogReg AUC)

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

y_train = (train_df[TARGET_COL] == 1).astype(int).values
y_test = (test_df[TARGET_COL] == 1).astype(int).values

lr_std = LogisticRegression(max_iter=500, random_state=42).fit(X_train, y_train)
auc_std = roc_auc_score(y_test, lr_std.predict_proba(X_test)[:, 1])

scaler_mm = MinMaxScaler()
X_train_mm = scaler_mm.fit_transform(train_df[FEATURES_SELECTED].fillna(0))
X_test_mm = scaler_mm.transform(test_df[FEATURES_SELECTED].fillna(0))
lr_mm = LogisticRegression(max_iter=500, random_state=42).fit(X_train_mm, y_train)
auc_mm = roc_auc_score(y_test, lr_mm.predict_proba(X_test_mm)[:, 1])

print('LogReg AUC (test):')
print(f'  StandardScaler: {auc_std:.4f}')
print(f'  MinMaxScaler:   {auc_mm:.4f}')
print('Выбран: StandardScaler' if auc_std >= auc_mm else 'Выбран: MinMaxScaler')

LogReg AUC (test):
  StandardScaler: 0.6270
  MinMaxScaler:   0.6070
Выбран: StandardScaler


## 5. Сохранение scaler и данных

In [5]:
import joblib

os.makedirs(os.path.join(_root, 'outputs'), exist_ok=True)
os.makedirs(os.path.join(_root, 'models'), exist_ok=True)

scaler_path = os.path.join(_root, 'models', 'scaler_tp_sl_1_05.joblib')
joblib.dump({'scaler': scaler, 'features': FEATURES_SELECTED, 'target': TARGET_COL}, scaler_path)

# Сохраняем scaled данные для 08 (опционально — 08 может пересчитать)
train_df_scaled = train_df.copy()
train_df_scaled[[f + '_scaled' for f in FEATURES_SELECTED]] = X_train
print(f'Сохранено: {scaler_path}')

Сохранено: c:\project\trading_bot_2Engine\models\scaler_tp_sl_1_05.joblib


## Итог шага 5

- StandardScaler fit на train, transform на train/val/test
- Проверка: mean≈0, std≈1 на train
- Сравнение с MinMaxScaler по AUC LogReg
- Сохранён scaler в `models/scaler_tp_sl_1_05.joblib` (features + scaler + target) для inference и 09_Model_Training